# NR7 + Inside Bar Breakout on SPY
## Strategy Brief
The NR7 + Inside Bar Breakout strategy combines two patterns to identify potential breakout opportunities. NR7 refers to the narrowest range of the last seven trading days, indicating a period of consolidation. An Inside Bar occurs when the current day's high and low are within the previous day's range, suggesting indecision. The strategy predicts that a breakout from this pattern may lead to a significant price move. Trades are executed when the price breaks out of the Inside Bar's range, with the expectation of capturing the ensuing trend.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
Define the parameters and constants required for the strategy.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SYMBOL = 'SPY'
NR_PERIOD = 7

### PHASE 2 - Data Exploration
Download historical data for SPY and compute the NR7 and Inside Bar indicators.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(SYMBOL, start=START_DATE, end=END_DATE)
data['Range'] = data['High'] - data['Low']

# Calculate NR7
nr7 = data['Range'].rolling(window=NR_PERIOD).min() == data['Range']
data['NR7'] = nr7

# Calculate Inside Bar
inside_bar = (data['High'] <= data['High'].shift(1)) & (data['Low'] >= data['Low'].shift(1))
data['InsideBar'] = inside_bar

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.scatter(data.index, data['Close'][data['NR7']], color='red', label='NR7', marker='^')
plt.scatter(data.index, data['Close'][data['InsideBar']], color='blue', label='Inside Bar', marker='o')
plt.title('SPY Price with NR7 and Inside Bar')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
Define the signal for entry and exit based on NR7 and Inside Bar breakout.

In [ ]:
# Signal for breakout
breakout_up = (data['NR7'] & data['InsideBar']) & (data['Close'] > data['High'].shift(1))
breakout_down = (data['NR7'] & data['InsideBar']) & (data['Close'] < data['Low'].shift(1))

# Positions: 1 for long, -1 for short, 0 for no position
data['Position'] = 0
data.loc[breakout_up, 'Position'] = 1
data.loc[breakout_down, 'Position'] = -1

### PHASE 4 - Coding & Backtesting
Backtest the strategy using the defined signals and plot the equity curve.

In [ ]:
# Shift positions for backtesting
positions = data['Position'].shift(1).fillna(0)

daily_returns = data['Close'].pct_change().fillna(0)
strategy_returns = positions * daily_returns

# Equity curve
equity_curve = (1 + strategy_returns).cumprod()

plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Equity Curve of NR7 + Inside Bar Breakout Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
Evaluate the performance of the strategy using key financial metrics.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve[-1] ** (252.0 / len(returns))) - 1
    sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)
    downside_returns = returns[returns < 0]
    sortino_ratio = returns.mean() / downside_returns.std() * np.sqrt(252)
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown = calculate_performance_metrics(strategy_returns)

# Buy and hold comparison
data['BuyHold'] = (1 + daily_returns).cumprod()
bh_cagr = (data['BuyHold'][-1] ** (252.0 / len(data))) - 1

print(f"Strategy CAGR: {cagr:.2%}")
print(f"Strategy Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Strategy Sortino Ratio: {sortino_ratio:.2f}")
print(f"Strategy Calmar Ratio: {calmar_ratio:.2f}")
print(f"Strategy Max Drawdown: {max_drawdown:.2%}")
print(f"Buy and Hold CAGR: {bh_cagr:.2%}")

### PHASE 6 - Deploy & Monitor
Create a function to download recent data, compute today's signal, and print the position.

In [ ]:
def check_today_signal():
    recent_data = yf.download(SYMBOL, start=pd.Timestamp.today() - pd.Timedelta(days=60), end=pd.Timestamp.today())
    recent_data['Range'] = recent_data['High'] - recent_data['Low']
    nr7 = recent_data['Range'].rolling(window=NR_PERIOD).min() == recent_data['Range']
    inside_bar = (recent_data['High'] <= recent_data['High'].shift(1)) & (recent_data['Low'] >= recent_data['Low'].shift(1))
    breakout_up = (nr7 & inside_bar) & (recent_data['Close'] > recent_data['High'].shift(1))
    breakout_down = (nr7 & inside_bar) & (recent_data['Close'] < recent_data['Low'].shift(1))
    position = 0
    if breakout_up.iloc[-1]:
        position = 1
    elif breakout_down.iloc[-1]:
        position = -1
    print(f"Today's Position: {position}")

check_today_signal()